## Shelters and Reachability

Attach shelters and validate the graph connectivity and reachability

In [ ]:
import osmnx as ox
import pandas as pd
import geopandas as gpd
import networkx as nx

In [ ]:
def load_shelters(csv_path):
    """
    Load shelters from CSV and return GeoDataFrame (EPSG:4326).
    Expected columns: name, lat, lon
    """
    shelters = pd.read_csv(csv_path)

    required_cols = {"name", "lat", "lon"}
    missing = required_cols - set(shelters.columns)
    if missing:
        raise ValueError(f"Shelter file missing columns: {missing}")

    shelters["lat"] = pd.to_numeric(shelters["lat"], errors="coerce")
    shelters["lon"] = pd.to_numeric(shelters["lon"], errors="coerce")
    shelters = shelters.dropna(subset=["lat", "lon"])

    shelters_gdf = gpd.GeoDataFrame(
        shelters,
        geometry=gpd.points_from_xy(shelters["lon"], shelters["lat"]),
        crs="EPSG:4326",
    )

    print(f"[Shelters] Loaded {len(shelters_gdf)} valid shelters")

    return shelters_gdf

In [ ]:
shelters_gdf = load_shelters("../data/raw/shelter/hatyai_shelters.csv")
shelters_gdf.to_file(
    "../data/processed/hatyai_shelters.geojson",
    driver="GeoJSON"
)

In [ ]:
def attach_shelters_to_nodes(graph_proj, shelters_gdf):
    """
    Snap shelters to nearest OSMnx nodes using OSMnx native method.
    """
    shelters_proj = shelters_gdf.to_crs(graph_proj.graph["crs"])
    xs = shelters_proj.geometry.x
    ys = shelters_proj.geometry.y

    nearest_nodes = ox.distance.nearest_nodes(graph_proj, X=xs, Y=ys)

    shelter_map = {}
    for node_id, shelter_name in zip(nearest_nodes, shelters_gdf["name"]):
        shelter_map.setdefault(node_id, []).append(shelter_name)

    nodes, edges = ox.graph_to_gdfs(graph_proj)

    nodes["is_shelter"] = False
    nodes["shelter_names"] = ""

    for node_id, names in shelter_map.items():
        if node_id in nodes.index:
            nodes.loc[node_id, "is_shelter"] = True
            nodes.loc[node_id, "shelter_names"] = "; ".join(sorted(set(names)))

    print(f"[Shelters] Attached to {nodes['is_shelter'].sum()} nodes")

    return nodes, edges

In [ ]:
graph = ox.load_graphml("../data/processed/hatyai_graph_with_capacity.graphml")

graph_proj = ox.project_graph(graph)
nodes, edges = attach_shelters_to_nodes(graph_proj, shelters_gdf)
graph_updated  = ox.graph_from_gdfs(nodes, edges)

# Project back to lat/lon before saving
graph_updated = ox.project_graph(graph_updated, to_crs="EPSG:4326")
ox.save_graphml(graph_updated, "../data/processed/hatyai_graph_with_shelters.graphml")

In [ ]:
print(edges.columns)
print(edges.index.names)

Optionally mirror edges to allow reverse travel in reachability checks

In [ ]:
MAKE_EDGES_BIDIR = True

edges_reset = edges.reset_index()
if MAKE_EDGES_BIDIR:
    reversed_edges = edges_reset.copy()
    reversed_edges[["u", "v"]] = reversed_edges[["v", "u"]]

    edges_use = (
        pd.concat([edges_reset, reversed_edges], ignore_index=True)
        .drop_duplicates(subset=["u", "v", "key"])
    )
else:
    edges_use = edges_reset

print("edges_use rows:", len(edges_use))

In [ ]:
def check_static_connectivity(nodes_df, edges_df):
    """
    Check structural connectivity of the graph.
    """

    G = nx.from_pandas_edgelist(
        edges_df,
        source="u",
        target="v",
        create_using=nx.DiGraph()
    )

    if G.number_of_nodes() == 0:
        print("[WARNING] Graph is empty.")
        return None

    G_u = G.to_undirected()

    n_comp = nx.number_connected_components(G_u)
    largest = max(nx.connected_components(G_u), key=len)
    coverage = len(largest) / G.number_of_nodes()

    print("------ Connectivity Report ------")
    print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")
    print(f"Connected components: {n_comp}")
    print(f"Largest component coverage: {coverage:.3f}")

    isolates = list(nx.isolates(G))
    print(f"Isolates: {len(isolates)}")

    return {
        "n_components": n_comp,
        "coverage": coverage,
        "isolates": isolates
    }

In [ ]:
def check_reachability_to_shelters(nodes_df, edges_df):
    """
    Check if all nodes can reach at least one shelter.
    """

    if "is_shelter" not in nodes_df.columns:
        raise ValueError("Nodes must contain 'is_shelter' column")

    shelter_nodes = set(nodes_df.index[nodes_df["is_shelter"]])

    if not shelter_nodes:
        print("[WARNING] No shelters found in nodes.")
        return None

    G_dir = nx.from_pandas_edgelist(
        edges_df,
        source="u",
        target="v",
        edge_attr="length",
        create_using=nx.DiGraph(),
    )

    G_rev = G_dir.reverse(copy=False)

    dist = nx.multi_source_dijkstra_path_length(
        G_rev,
        list(shelter_nodes),
        weight="length"
    )

    reachable = set(dist.keys())
    all_nodes = set(nodes_df.index)
    unreachable = all_nodes - reachable

    print("------ Reachability Report ------")
    print(f"Shelters: {len(shelter_nodes)}")
    print(f"Reachable nodes: {len(reachable)}")
    print(f"Unreachable nodes: {len(unreachable)}")

    return {
        "reachable": reachable,
        "unreachable": unreachable,
        "distances": dist
    }

In [ ]:
check_static_connectivity(nodes, edges_use)
check_reachability_to_shelters(nodes, edges_use)